In [1]:
from stable_baselines3 import PPO, TD3
from sb3_contrib import TQC, RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from rl4greencrab import evaluate_agent, multiConstAction, simulator
import pandas as pd
import numpy as np
from rl4greencrab import plot_agent
import ray
from skopt import gp_minimize, gbrt_minimize 
from skopt.plots import plot_convergence, plot_objective
from rl4greencrab import TwoActNormalized, twoActEnv

## Simulate agent actions in the environment

In [3]:
obs = "count"
config = {
    'random_start':True,
    'var_penalty_const': 0,
    'observation_type': obs,
    'control_randomness': True
}

In [3]:
dir_name = 'greencrab_two_act_env/count'

### Constant Action

In [ ]:
# create instance of env
evalEnv =  twoActEnv(config)

# read in GP-optimized constant action
x = pd.read_csv("../data/constant_actions_result.csv").values[0].tolist()

# turn action into agent
constant_agent = multiConstAction(env=evalEnv, action=np.array(x))
const_plot_agent = plot_agent(env_sim_df=None, 
                              agent_name='const_agent', 
                              env=evalEnv, 
                              agent=constant_agent, 
                              save_dir='.')

# run 500 simulations of environment and constant action "agent"                             
df = const_plot_agent.gen_env_sim_df(rep=500, obs_names=['crabs'])

# get just final timestep
df_subset = df[df['t'] == 99]

# save df
df_subset.to_csv('../data/const_agent_simulations.csv', index=False)

### RL Policy

In [ ]:
evalEnv =  TwoActNormalized(config)
ppoAgent = PPO.load(f"{agent_dir}/PPO-Var0-({obs})-1", device="cpu")
ppo_plot_agent = plot_agent(env_sim_df=None, 
                            agent_name=f'{dir_name}/ppo_agent_size', 
                            env=evalEnv, 
                            agent=ppoAgent, 
                            save_dir='.')
df = ppo_plot_agent.gen_env_sim_df(rep=500, obs_names=['crabs'])
ppo_plot_agent.save_df(ppo_plot_agent.env_simulation_df, 'ppo_size_sim_500')